# Análise de Entropia das Saídas Criptográficas

Este notebook calcula e visualiza a **entropia de Shannon** (bits/byte) das saídas
de diferentes funções criptográficas, todas recebendo a **mesma senha de entrada**.

O objetivo é mostrar como cada técnica transforma uma entrada de **baixa entropia**
(senha fraca) em uma saída de **alta entropia** (próxima de 8 bits/byte — máximo teórico).

**Entrada fixa para todos os métodos:**
- Senha: `senha_fraca`
- Salt: `salt_fixo_aula`
- Chave HMAC: `chave_hmac_aula`

**Métodos analisados:**
| Grupo | Método | Tamanho da saída |
|---|---|---|
| Hash | MD5 | 128 bits |
| Hash | SHA-1 | 160 bits |
| Hash | SHA-2 (256) | 256 bits |
| Hash | SHA-3 (256) | 256 bits |
| Autenticação | HMAC-SHA256 | 256 bits |
| Proteção de senha | SHA-256 + Salt | 256 bits |
| KDF | PBKDF2-HMAC-SHA256 | 256 bits |
| KDF | Scrypt | 256 bits |

> **Como calculamos a entropia:**  
> Cada função é executada **N=10 vezes** com entradas ligeiramente variadas
> (`senha_fraca_0`, `senha_fraca_1`, ...) para acumular bytes suficientes e estimar
> a distribuição de forma estável. A fórmula é:
> $$H = -\sum_{b=0}^{255} p_b \log_2 p_b $$
> O máximo teórico é **8 bits/byte** (distribuição uniforme sobre 256 valores).

In [ ]:
# !pip install matplotlib numpy mnemonic eth-keys

In [ ]:
import hashlib
import hmac as hmac_lib
import math
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# BIP-39
try:
    from mnemonic import Mnemonic
    BIP39_OK = True
except ImportError:
    BIP39_OK = False
    print("[aviso] mnemonic não instalado — célula BIP-39 será ignorada.")
    print("        Execute: pip install mnemonic")

## Tabela de Conteúdos

- [Análise de Entropia das Saídas Criptográficas](#análise-de-entropia-das-saídas-criptográficas)
  - [Parâmetros](#parametros)
  - [Métodos](#metodos)
  - [Experimentos](#experimentos)
    - [Funções Hash](#funções-hash)
    - [HMAC-SHA256 e SHA-256 + Salt](#hmac-sha256-e-sha-256--salt)
    - [KDFs — PBKDF2 e Scrypt](#3-kdfs--pbkdf2-e-scrypt)
    - [Entrada de referência](#entrada-de-referência)
  - [Comparação dos Métodos](#comparação-dos-métodos)


## Parametros

In [ ]:
SENHA      = b"senha_fraca"
SALT       = b"salt_fixo_aula"
CHAVE_HMAC = b"chave_hmac_aula"

N_HASH = 10   # iterações para Hash, HMAC e Salt  
N_KDF  = 500     # iterações para KDFs               

## Metodos

In [ ]:
# Calcula a entropia de Shannon em bits/byte sobre a distribuição
def entropia_bytes(data: bytes) -> float:
    
    n = len(data)
    if n == 0:
        return 0.0
    freq = Counter(data)
    return -sum((c / n) * math.log2(c / n) for c in freq.values())


# número total de bits no dado
def bits_totais(data: bytes) -> int:
    return len(data) * 8


print(f"Entropia de zeros:         {entropia_bytes(bytes(32)):.4f} bits/byte")
print(f"Entropia de bytes aleat.:  {entropia_bytes(bytes(range(256))*4):.4f} bits/byte")

## Experimentos


### Funções Hash


MD5, SHA-1, SHA-256 e SHA3-256 recebem `senha_fraca_i` e produzem
um digest de tamanho fixo. Apesar da entrada fraca, a saída deve ter
entropia próxima de **8 bits/byte**

In [ ]:
hashes = {
    "MD5":      hashlib.md5,
    "SHA-1":    hashlib.sha1,
    "SHA-256":  hashlib.sha256,
    "SHA3-256": hashlib.sha3_256,
}

resultados_hash = {}
for nome, fn in hashes.items():
    buf = b"".join(
        fn(SENHA + str(i).encode()).digest()
        for i in range(N_HASH)
    )
    resultados_hash[nome] = {
        "bytes":       buf,
        "entropia":    entropia_bytes(buf),
        "bits_saida":  fn(SENHA).digest().__len__() * 8,   # bits de UMA saída
        "amostras":    len(buf),
    }
    print(f"{nome:10s}: entropia = {resultados_hash[nome]['entropia']:.4f} bits/byte "
          f"| saída = {resultados_hash[nome]['bits_saida']} bits")

### HMAC-SHA256 e SHA-256 + Salt

- **HMAC-SHA256** adiciona uma chave secreta à função hash, protege contra ataques de extensão de comprimento.
- **SHA-256 + Salt** concatena um salt antes de aplicar o hash, impede tabelas rainbow.

Ambos produzem 256 bits. A entropia da saída deve ser igualmente alta.

In [ ]:
# HMAC-SHA256 
buf_hmac = b"".join(
    hmac_lib.new(CHAVE_HMAC, SENHA + str(i).encode(), hashlib.sha256).digest()
    for i in range(N_HASH)
)

# SHA-256 + Salt 
buf_salt = b"".join(
    hashlib.sha256(SALT + SENHA + str(i).encode()).digest()
    for i in range(N_HASH)
)

resultados_auth = {
    "HMAC-SHA256": {
        "bytes":      buf_hmac,
        "entropia":   entropia_bytes(buf_hmac),
        "bits_saida": 256,
    },
    "SHA256+Salt": {
        "bytes":      buf_salt,
        "entropia":   entropia_bytes(buf_salt),
        "bits_saida": 256,
    },
}

for nome, r in resultados_auth.items():
    print(f"{nome:12s}: entropia = {r['entropia']:.4f} bits/byte | saída = {r['bits_saida']} bits")

## 3. KDFs — PBKDF2 e Scrypt

As KDFs são projetadas para ser **lentas e custosas** — dificultam ataques de
força bruta. Usamos `N_KDF=50` amostras (em vez de 1000) para não demorar.
Com menos amostras a estimativa de entropia é ligeiramente menos precisa,
mas suficiente para ilustrar o comportamento.

In [ ]:
#  PBKDF2-HMAC-SHA256 
print(f"Calculando PBKDF2 ({N_KDF} amostras × 100.000 iterações)...")
buf_pbkdf2 = b"".join(
    hashlib.pbkdf2_hmac(
        "sha256",
        SENHA + str(i).encode(),
        SALT,
        100_000,
        dklen=32
    )
    for i in range(N_KDF)
)

#  Scrypt 
print(f"Calculando Scrypt  ({N_KDF} amostras × N=8192)...")
buf_scrypt = b"".join(
    hashlib.scrypt(
        SENHA + str(i).encode(),
        salt=SALT,
        n=8192, r=8, p=1,
        dklen=32
    )
    for i in range(N_KDF)
)

resultados_kdf = {
    "PBKDF2": {
        "bytes":      buf_pbkdf2,
        "entropia":   entropia_bytes(buf_pbkdf2),
        "bits_saida": 256,
    },
    "Scrypt": {
        "bytes":      buf_scrypt,
        "entropia":   entropia_bytes(buf_scrypt),
        "bits_saida": 256,
    },
}

for nome, r in resultados_kdf.items():
    print(f"{nome:8s}: entropia = {r['entropia']:.4f} bits/byte | saída = {r['bits_saida']} bits")

### Entrada de referência

Calculamos a entropia da **entrada bruta** (`senha_fraca`) para usar como
referência. 

MOstrando o ponto de partida antes de qualquer transformação criptográfica.

In [ ]:
buf_entrada = b"".join(
    (SENHA + str(i).encode())
    for i in range(N_HASH)
)

entropia_entrada = entropia_bytes(buf_entrada)
bits_entrada     = len(SENHA) * 8   # bits de UMA senha

print(f"Entrada (senha_fraca): entropia = {entropia_entrada:.4f} bits/byte")
print(f"Tamanho de uma senha:  {bits_entrada} bits")

## Comparação dos Métodos

**Gráfico 1** :  Mostra a **qualidade** da saída de cada método.  
A linha tracejada em **8 bits/byte** representa o máximo teórico.

**Gráfico 2** : Mostra o **aumento de bits** — quantos bits de saída cada método produz
a partir dos mesmos 88 bits de entrada (`senha_fraca`).

In [ ]:
# tabela
todos = {
    "Entrada\n(senha_fraca)": {"entropia": entropia_entrada, "bits_saida": bits_entrada,  "grupo": "entrada"},
    "MD5":                    {**resultados_hash["MD5"],      "grupo": "hash"},
    "SHA-1":                  {**resultados_hash["SHA-1"],    "grupo": "hash"},
    "SHA-256":                {**resultados_hash["SHA-256"],  "grupo": "hash"},
    "SHA3-256":               {**resultados_hash["SHA3-256"], "grupo": "hash"},
    "HMAC-SHA256":            {**resultados_auth["HMAC-SHA256"], "grupo": "autenticação"},
    "SHA256\n+Salt":          {**resultados_auth["SHA256+Salt"],  "grupo": "salt"},
    "PBKDF2":                 {**resultados_kdf["PBKDF2"],    "grupo": "kdf"},
    "Scrypt":                 {**resultados_kdf["Scrypt"],    "grupo": "kdf"},
}

nomes    = list(todos.keys())
entropias = [todos[n]["entropia"]   for n in nomes]
bits      = [todos[n]["bits_saida"] for n in nomes]
grupos    = [todos[n]["grupo"]       for n in nomes]

CORES = {
    "entrada":      "#e74c3c",
    "hash":         "#3498db",
    "autenticação": "#9b59b6",
    "salt":         "#27ae60",
    "kdf":          "#f39c12",
}
cores = [CORES[g] for g in grupos]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "Efeito das Técnicas Criptográficas na Entropia da Senha",
    fontsize=15, fontweight="bold", y=1.02
)

x = np.arange(len(nomes))
BAR_W = 0.55

bars1 = ax1.bar(x, entropias, width=BAR_W, color=cores, edgecolor="white", linewidth=0.8)
ax1.axhline(8.0, color="black", linestyle="--", linewidth=1.2, label="Máximo teórico (8 bits/byte)")
ax1.axhline(entropias[0], color=CORES["entrada"], linestyle=":",
            linewidth=1.2, label=f"Entrada ({entropias[0]:.2f} bits/byte)")

for bar, val in zip(bars1, entropias):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.05,
        f"{val:.2f}",
        ha="center", va="bottom", fontsize=8, fontweight="bold"
    )

ax1.set_xticks(x)
ax1.set_xticklabels(nomes, fontsize=9)
ax1.set_ylim(0, 8.8)
ax1.set_ylabel("Entropia (bits/byte)", fontsize=11)
ax1.set_title("Qualidade da Saída", fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(axis="y", alpha=0.3)
ax1.spines[["top", "right"]].set_visible(False)

# G2
bars2 = ax2.bar(x, bits, width=BAR_W, color=cores, edgecolor="white", linewidth=0.8)
ax2.axhline(bits[0], color=CORES["entrada"], linestyle=":",
            linewidth=1.2, label=f"Entrada ({bits[0]} bits)")

for bar, val in zip(bars2, bits):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        val + 2,
        f"{val}",
        ha="center", va="bottom", fontsize=8, fontweight="bold"
    )

ax2.set_xticks(x)
ax2.set_xticklabels(nomes, fontsize=9)
ax2.set_ylabel("Bits de saída", fontsize=11)
ax2.set_title("Aumento de Bits", fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(axis="y", alpha=0.3)
ax2.spines[["top", "right"]].set_visible(False)

# Legenda de grupos
patches = [mpatches.Patch(color=c, label=g.capitalize()) for g, c in CORES.items()]
fig.legend(handles=patches, loc="lower center", ncol=5,
           fontsize=9, title="Grupo", title_fontsize=9,
           bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.savefig("entropia_comparacao.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva em entropia_comparacao.png")